In [1]:
import pandas as pd
from gensim import corpora, models
from tqdm import tqdm

tqdm.pandas()

In [2]:
INPUT_CSV = "dataset/clean.csv"

In [3]:
dataframe = pd.read_csv(INPUT_CSV, usecols=["clean_speech"])

In [4]:
size = len(dataframe)

In [5]:
def token_stream():
    for speech in dataframe.clean_speech:
        tokens = speech.split()
        yield tokens

In [6]:
dictionary = corpora.Dictionary(prune_at=None)

for tokens in tqdm(token_stream(), total=size):
    dictionary.add_documents([tokens])
dictionary.filter_extremes(no_above=0.5)
# dictionary.compactify()
dictionary.save("models/greek.dict")

100%|██████████| 1184741/1184741 [01:18<00:00, 15178.23it/s]


In [7]:
class BowCorpus:
    def __init__(self, dictionary):
        self.dictionary = dictionary

    def __iter__(self):
        for tokens in token_stream():
            yield self.dictionary.doc2bow(tokens)

    def __getitem__(self, index):
        tokens = dataframe.iloc[index]
        return self.dictionary.doc2bow(tokens)

In [8]:
corpus = BowCorpus(dictionary)
corpora.MmCorpus.serialize("models/greek_corpus.mm", corpus)

In [9]:
tfidf_model = models.TfidfModel(corpus)
tfidf_model.save("models/greek.tfidf")

lsi_model = models.LsiModel(corpus, id2word=dictionary, random_seed=42)
lsi_model.save("models/greek.lsi")

In [10]:
tfidf2 = models.TfidfModel.load("models/greek.tfidf")
lsi2 = models.LsiModel.load("models/greek.lsi")

In [11]:
lsi2[tfidf2[dictionary.doc2bow("δημοκρατία του λαού".split())]]

[(np.int64(0), np.float64(0.08445516510375228)),
 (np.int64(1), np.float64(0.019313906419495548)),
 (np.int64(2), np.float64(0.01259600187382138)),
 (np.int64(3), np.float64(-0.00011260479220144051)),
 (np.int64(4), np.float64(-0.08820464698653606)),
 (np.int64(5), np.float64(0.032053202186226176)),
 (np.int64(6), np.float64(0.12348902876282034)),
 (np.int64(7), np.float64(0.16455475627227528)),
 (np.int64(8), np.float64(0.18747791383674323)),
 (np.int64(9), np.float64(-0.005895534924536297)),
 (np.int64(10), np.float64(0.18762238475907306)),
 (np.int64(11), np.float64(-0.0038793665963254894)),
 (np.int64(12), np.float64(-0.0688430614504985)),
 (np.int64(13), np.float64(0.2300171645714204)),
 (np.int64(14), np.float64(0.09806005480018545)),
 (np.int64(15), np.float64(0.09972408370354606)),
 (np.int64(16), np.float64(-0.02125351779361754)),
 (np.int64(17), np.float64(-0.15996828670987318)),
 (np.int64(18), np.float64(-0.03864325756459807)),
 (np.int64(19), np.float64(0.02579124037967856